In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install rdkit==2024.9.5
!pip install torch_geometric==2.5.3

In [2]:
import os
import sys
doc_name = "/content/drive/MyDrive/BHRGNN-Code"
sys.path.append(doc_name)
import datetime
import torch
import pandas as pd
import numpy as np
import csv
import random
from tqdm import tqdm
from utils.BRG import *
from utils.FPT import *
from torch_geometric.nn import GATConv
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import global_mean_pool as gap
from torch_geometric.nn import global_max_pool as gmp
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Visualize of connection type

In [3]:
# Connection type
connection_type_dict = {
    'Chain': torch.tensor([[0,0,1,1,2,2,3,3,4,4,1,3,2],
                           [0,2,1,4,2,3,3,1,4,1,3,2,0]], dtype=torch.long),
    'Cycle': torch.tensor([[0,0,0,1,1,2,2,3,3,4,4,4,1,3,2],
                           [0,2,4,1,4,2,3,3,1,4,1,0,3,2,0]], dtype=torch.long),
    'Tree': torch.tensor([[0,0,1,1,2,2,3,3,4,4,4,4,4],
                          [0,4,1,4,2,4,3,4,0,1,2,3,4]], dtype=torch.long),
    'Star': torch.tensor([[0,0,0,0,1,1,1,1,2,2,2,2,3,3,3,3,4,4,4,4,4],
                          [0,1,2,4,1,0,3,4,2,0,3,4,3,1,2,4,4,0,1,2,3]], dtype=torch.long),
    'Complete': torch.tensor([[0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 4, 4, 4, 4,4],
                              [0, 1, 2, 3, 4, 0, 1, 2, 3, 4, 0, 1, 2, 3, 4, 0, 1, 2, 3, 4, 0, 1, 2, 3,4]], dtype=torch.long)
}

# import data
files = ['Aryl_halide_new.csv', 'Product_new.csv', 'Base_new.csv', 'Ligand_new.csv', 'Additive_new.csv']
for i in range(len(files)):
  files[i] = doc_name + "/data/" + files[i]
features = load_and_preprocess_data(files)
labels = pd.read_csv('%s/data/Buchwald.csv' % doc_name).iloc[:, -1].values / 100

Visualization = False
if Visualization:
  for key in connection_type_dict.keys():
    # Visualization
    data = Data(edge_index=connection_type_dict[key])
    visualize_graph_data(data=data, num_nodes=5, connect_type=key, custom_positions={0: (0, 1),1: (2, 1),2: (0.5, 0),3: (1.5, 0),4: (1, 2)}, node_labels={0: 'Aryl',1: ' Product',2: 'Base',3: 'Ligand',4: 'Additive'})

# GNN model framework

In [4]:
class Net(nn.Module):
    def __init__(self, in_channels):
        super(Net, self).__init__()
        self.conv1 = GATConv(in_channels, 512, heads=2)
        self.norm1 = nn.BatchNorm1d(1024)
        self.conv2 = GATConv(1024, 256, heads=2)
        self.norm2 = nn.BatchNorm1d(512)

        self.lin1 = nn.Linear(3072, 1024)
        self.lin2 = nn.Linear(1024, 512)
        self.lin3 = nn.Linear(512, 256)
        self.lin4 = nn.Linear(256, 128)
        self.lin5 = nn.Linear(128, 1)
        self.bn1 = nn.BatchNorm1d(3072)
        self.bn2 = nn.BatchNorm1d(1024)
        self.bn3 = nn.BatchNorm1d(512)
        self.bn4 = nn.BatchNorm1d(256)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index)
        x = self.norm1(x)
        x1 = F.relu(x)

        x = self.conv2(x1, edge_index)
        x = self.norm2(x)
        x2 = F.relu(x)

        x = torch.cat([x1, x2], 1)

        x_mean = gap(x, batch=batch)
        x_max = gmp(x, batch=batch)
        x = torch.cat([x_mean, x_max], 1)

        x = x.view(x.shape[0], -1)

        x = self.bn1(x)
        x = self.lin1(x)
        x = nn.Dropout(p=0.5)(x)
        x = nn.ReLU()(x)

        x = self.bn2(x)
        x = self.lin2(x)
        x = nn.ReLU()(x)

        x = self.bn3(x)
        x = self.lin3(x)
        x = nn.ReLU()(x)

        x = self.bn4(x)
        x = self.lin4(x)
        x = nn.ReLU()(x)

        x = self.lin5(x)

        return x.squeeze()

# Model training

In [5]:
def train_model(connection_type:str,
         edge_index:list,
         report_addr:str,
         rs:int=42,
         lr:float=1e-4,
         batch_size:int=64,
         num_epochs:int=2000,
         reture_metrics:bool=True,
         ):
  # random state
  random.seed(rs)
  torch.manual_seed(rs)

  # Dataset
  datas = create_all_graphs(features, edge_index)
  dataset = CustomGraphDataset(datas, labels)

  # dataset splitting
  train_size = int(0.7 * len(dataset))
  val_size = len(dataset) - train_size
  train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

  # DataLoader
  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
  val_loader = DataLoader(val_dataset, batch_size=val_size)
  print(f'{connection_type} trainning data generated.')

  # model initialization
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model = Net(100).to(device)
  opti = torch.optim.Adam(model.parameters(), lr=1e-4)
  criterion = torch.nn.MSELoss()

  # report
  dir_path = "%s/model_training/BHRGNN_%s_rs=%s_%s" % (report_addr, connection_type, rs, datetime.datetime.now())
  os.mkdir("%s" % dir_path)
  metrics_path = '%s/metrics.csv' % dir_path

  # evaluarion metrics
  metrics_names = ['Epoch', 'Train_Loss', 'Test_Loss', 'Train_RMSE', 'Test_RMSE', 'Train_R2', 'Test_R2', 'Train_MAE', 'Test_MAE']
  train_losses = []
  test_losses = []
  maes_train = []
  rmses_train = []
  r2s_train = []
  maes_test = []
  rmses_test = []
  r2s_test = []

  with open(metrics_path, 'w', newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=metrics_names)
    writer.writeheader()

  print(f'Traning GNN for {connection_type}:')

  # train
  for epoch in tqdm(range(num_epochs), desc=f"Epochs for {connection_type} Model rs={rs}"):
    model.train()
    running_loss_train = 0.0
    all_preds_train = []
    all_targets_train = []

    for data in train_loader:
        out = model(data[0].to(device))
        loss = criterion(out, data[1].to(device))
        opti.zero_grad()
        loss.backward()
        opti.step()
        running_loss_train += loss.item()

        # save predicted value and ground truth for evaluation metrics calculation
        all_preds_train.append(out.detach().cpu().numpy())
        all_targets_train.append(data[1].detach().cpu().numpy())

    epoch_loss_train = running_loss_train / len(train_loader)

    # Convert to numpy
    all_preds_train = np.concatenate(all_preds_train)
    all_targets_train = np.concatenate(all_targets_train)

    # evaluation metrics for trainset
    mae_train = mean_absolute_error(all_targets_train, all_preds_train)
    rmse_train = np.sqrt(np.mean((all_preds_train - all_targets_train) ** 2))
    r2_train = r2_score(all_targets_train, all_preds_train)

    # test
    with torch.no_grad():
        running_loss_val = 0.0
        all_preds_val = []
        all_targets_val = []

        for data in val_loader:
            out = model(data[0].to(device))
            val_loss = criterion(out, data[1].to(device))
            running_loss_val += val_loss.item()

            # save predicted value and ground truth
            all_preds_val.append(out.detach().cpu().numpy())
            all_targets_val.append(data[1].detach().cpu().numpy())

        epoch_loss_val = running_loss_val / len(val_loader)
        test_losses.append(epoch_loss_val)

        # Convert to numpy
        all_preds_val = np.concatenate(all_preds_val)
        all_targets_val = np.concatenate(all_targets_val)

        # evaluation metrics for testset
        mae_val = mean_absolute_error(all_targets_val, all_preds_val)
        rmse_val = np.sqrt(np.mean((all_preds_val - all_targets_val) ** 2))
        r2_val = r2_score(all_targets_val, all_preds_val)

    # the record of evaluation metrics for this round
    train_losses.append(epoch_loss_train)
    maes_train.append(mae_train)
    rmses_train.append(rmse_train)
    r2s_train.append(r2_train)
    maes_test.append(mae_val)
    rmses_test.append(rmse_val)
    r2s_test.append(r2_val)

    with open(metrics_path, 'a', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=metrics_names)
        writer.writerow({
            'Epoch': epoch + 1,
            'Train_Loss': epoch_loss_train,
            'Test_Loss': epoch_loss_val,
            'Train_RMSE': rmse_train,
            'Test_RMSE': rmse_val,
            'Train_R2': r2_train,
            'Test_R2': r2_val,
            'Train_MAE': mae_train,
            'Test_MAE': mae_val
        })
  print(f'Finish training for {connection_type} B-H graph')
  # best performance
  best_index = r2s_test.index(max(r2s_test))

  # Plot of the model performance
  # R2
  plt.figure(figsize=(8, 5), dpi=200)
  plt.plot(range(1, num_epochs + 1), r2s_train, label='Trainset $R^2$')
  plt.plot(range(1, num_epochs + 1), r2s_test, label='Testset $R^2$', linestyle='--')
  plt.text(1000, 0.5, 'R$^2$=%.4f' % r2s_test[best_index], fontsize=14, va='center', ha='center')
  plt.title('Train/Test $R^2$ Over Epochs')
  plt.xlabel('Epoch')
  plt.ylabel('$R^2$')
  plt.legend()
  plt.savefig("%s/R2_performance.png" % dir_path)

  # MSE
  plt.figure(figsize=(8, 5), dpi=200)
  plt.plot(range(1, num_epochs + 1), train_losses, label='Trainset Loss')
  plt.plot(range(1, num_epochs + 1), test_losses, label='Testset Loss', linestyle='--')
  plt.text(1000, 0.05, 'MSE=%.4f' % test_losses[best_index], fontsize=14, va='center', ha='center')
  plt.title('Train/Test MSE Over Epochs')
  plt.xlabel('Epoch')
  plt.ylabel('MSE Loss')
  plt.legend()
  plt.savefig("%s/MSE_performance.png" % dir_path)

  # Output of the model best performance
  if reture_metrics:
    return {
        "train_R2":r2s_train[best_index],
        "train_RMSE":rmses_train[best_index],
        "train_MAE":maes_train[best_index],
        "test_R2":r2s_test[best_index],
        "test_RMSE":rmses_test[best_index],
        "test_MAE":maes_test[best_index]
    }

In [ ]:
rs_list = [1,2,3,4,5]
connection_type_list = ['Chain', 'Cycle', 'Tree', 'Star', 'Complete']

# Model training
for t in connection_type_list:

  # Evaluation metrics (total)
  eval_metrics = np.zeros((1+len(rs_list), 6))
  columns = ['train_R2', 'train_RMSE','train_MAE','test_R2','test_RMSE','test_MAE']
  index = []
  for rs in rs_list:
    index.append("%s" % rs)
  index.append("avg±std")
  eval_metrics = pd.DataFrame(eval_metrics, columns=columns, index=index)

  for m in range(len(rs_list)):
    metrics=train_model(connection_type=t,
         edge_index = connection_type_dict[t],
         report_addr = "%s" % doc_name,
         rs = rs_list[m],
         lr = 1e-4,
         batch_size = 64,
         num_epochs = 2,
         reture_metrics = True,
         )
    for name in eval_metrics.columns:
      eval_metrics.loc["%s" % rs_list[m]][name] = metrics[name]

  # Evaluation metrics report
  for i in range(len(rs_list), eval_metrics.shape[0], len(rs_list)+1):
    for j in range(eval_metrics.shape[1]):
      eval_metrics.iloc[i,j] = "%.4f ± %.4f" % (eval_metrics.iloc[i-len(rs_list):i-1,j].mean(), eval_metrics.iloc[i-len(rs_list):i-1,j].std())
  eval_metrics.to_csv("%s/model_training/BHRGNN-%s_report_%s.csv" % (doc_name, t, datetime.datetime.now()))
  print(eval_metrics)